# 04 – LSTM Model

This notebook trains a multi-layer LSTM network on the scaled, feature-engineered
dataset and evaluates it on the test set.


In [1]:
# ── 1. Imports & Configuration ──────────────────────────────────────────────
import sys

sys.path.insert(0, '..')

import warnings

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.data.preprocessor import DataPreprocessor
from src.models.lstm_model import LSTMModel
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR

plotter = Plotter()
SEQ_LEN = 10  # LSTM的时间步长（你可以根据你之前的设定调整，比如10或20）

In [2]:
# ── 2. Load Data & Scale ────────────────────────────────────────────────────
train = load_processed_data('train')
val = load_processed_data('val')
test = load_processed_data('test')

# 提取并保存真实的未归一化的收盘价，用于后面算误差
y_test_val_df = pd.concat([val, test])

# 归一化特征
pre = DataPreprocessor()
t_sc, v_sc, te_sc = pre.scale_features(train, val, test)

target_col = 'Close'
target_idx = list(train.columns).index(target_col)

print(f'Target column index: {target_idx}')
print(f'Scaled shapes – train: {t_sc.shape}, val: {v_sc.shape}, test: {te_sc.shape}')

Target column index: 1
Scaled shapes – train: (1006, 32), val: (252, 32), test: (249, 32)


In [3]:
# ── 3. Create LSTM Sequences ────────────────────────────────────────────────
import numpy as np
import pandas as pd


def create_sequences(data, target_index, seq_length):
    # 【修复点】：如果是 Pandas 格式，强制转换为 numpy 纯数组格式
    if isinstance(data, (pd.DataFrame, pd.Series)):
        data = data.values

    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i: i + seq_length])
        y.append(data[i + seq_length, target_index])
    return np.array(X), np.array(y)


# 传入数据进行切片
X_train, y_train = create_sequences(t_sc, target_idx, SEQ_LEN)
X_val, y_val = create_sequences(v_sc, target_idx, SEQ_LEN)
X_test, y_test = create_sequences(te_sc, target_idx, SEQ_LEN)

print(f"X_train shape: {X_train.shape} (Samples, TimeSteps, Features)")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (996, 10, 32) (Samples, TimeSteps, Features)
X_val shape:   (242, 10, 32)
X_test shape:  (239, 10, 32)


In [7]:
# ── 4. Train, Evaluate & Plot LSTM ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

lstm = LSTMModel()

val_idx = val.index[SEQ_LEN:] if len(val.index[SEQ_LEN:]) == len(X_val) else val.index[-len(X_val):]
test_idx = test.index[SEQ_LEN:] if len(test.index[SEQ_LEN:]) == len(X_test) else test.index[-len(X_test):]

print("--- Training LSTM (Two-Phase Refitting) ---")
# 1. 获取被归一化压缩的预测值
lstm_val_preds_scaled, lstm_test_preds_scaled = lstm.train_and_refit(
    X_train, y_train, X_val, y_val, X_test,
    val_index=val_idx, test_index=test_idx
)

# =========================================================================
# 📊 【新增】：绘制并保存训练 Loss 曲线
# =========================================================================
plt.figure(figsize=(10, 5))
plt.plot(lstm.history.history['loss'], label='Training Loss (MSE)', color='#1f77b4', linewidth=2)
plt.plot(lstm.history.history['val_loss'], label='Validation Loss (MSE)', color='#ff7f0e', linewidth=2)
plt.title('LSTM Refitting Phase - Loss Curve', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)

# 确保 figures 文件夹存在并保存
figures_dir = RESULTS_DIR.parent / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
loss_path = figures_dir / 'lstm_training_loss.png'
plt.savefig(loss_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"📉 Training Loss chart saved to: {loss_path}")

# =========================================================================
# 🔓 【核心修复】：反归一化 Inverse Transform (还原真实股价)
# =========================================================================
dummy_val = np.zeros((len(lstm_val_preds_scaled), len(train.columns)))
dummy_test = np.zeros((len(lstm_test_preds_scaled), len(train.columns)))

dummy_val[:, target_idx] = lstm_val_preds_scaled
dummy_test[:, target_idx] = lstm_test_preds_scaled

# 注意：这里假设你的预处理器里面归一化器叫做 scaler
# 如果报错说 DataPreprocessor 没有 scaler 属性，请根据你实际的变量名修改 (比如 pre.minmax_scaler)
val_preds_real = pre.scaler.inverse_transform(dummy_val)[:, target_idx]
test_preds_real = pre.scaler.inverse_transform(dummy_test)[:, target_idx]

lstm_val_preds = pd.Series(val_preds_real, index=val_idx, name='LSTM')
lstm_test_preds = pd.Series(test_preds_real, index=test_idx, name='LSTM')

# =========================================================================
# 📈 计算指标与保存
# =========================================================================
y_val_true = val.loc[val_idx, 'Close']
y_test_true = test.loc[test_idx, 'Close']

print("\n🏆 [Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(y_val_true, lstm_val_preds))

print("\n🏆 [Phase 2] 2025 Test Metrics:")
print(calculate_metrics(y_test_true, lstm_test_preds))

# 画预测对比图
plotter.plot_predictions_comparison(
    y_true=y_test_true,
    predictions={'LSTM Forecast': lstm_test_preds},
    dates=test_idx,
    title="LSTM 2025 Test Predictions",
    filename="lstm_test_forecast.png"
)

lstm.save("lstm_model.keras")
lstm_val_preds.to_csv(RESULTS_DIR / 'lstm_val_preds.csv')
lstm_test_preds.to_csv(RESULTS_DIR / 'lstm_test_preds.csv')
print("✅ LSTM execution entirely complete. All files and plots generated!")

--- Training LSTM (Two-Phase Refitting) ---
📉 Training Loss chart saved to: /Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/reports/figures/lstm_training_loss.png

🏆 [Phase 1] 2024 Validation Metrics:
{'mse': 460741.9566046062, 'rmse': 678.779755594262, 'mae': 523.1531208191128, 'mape': 2.617705383093303, 'directional_accuracy': 0.4730290456431535}

🏆 [Phase 2] 2025 Test Metrics:
{'mse': 1576289.5413448403, 'rmse': 1255.5037002513534, 'mae': 1057.5888977542259, 'mape': 4.491944463353144, 'directional_accuracy': 0.5126050420168067}
✅ LSTM execution entirely complete. All files and plots generated!


## Summary

LSTM model results are saved to `reports/results/lstm_metrics.json`.

Continue to **05_ensemble.ipynb** to combine all model predictions.